# MELD (dataset) + FLAN-T5 + Prompt Engineering

Este notebook prepara el dataset MELD, construye ventanas de contexto por diálogo y evalúa **prompt engineering** con **FLAN-T5** (sin entrenamiento adicional).

In [3]:
import kagglehub
import os
import pandas as pd

In [4]:
path = kagglehub.dataset_download("zaber666/meld-dataset")
print("Path to dataset files:", path)

Path to dataset files: C:\Users\Haojie\.cache\kagglehub\datasets\zaber666\meld-dataset\versions\1


In [5]:
# Cargar CSVs de texto
train_path = os.path.join(path+'/MELD-RAW/MELD.Raw/train/', "train_sent_emo.csv")
dev_path   = os.path.join(path+'/MELD-RAW/MELD.Raw/', "dev_sent_emo.csv")
test_path  = os.path.join(path+'/MELD-RAW/MELD.Raw/', "test_sent_emo.csv")

train = pd.read_csv(train_path, encoding="utf-8")
dev   = pd.read_csv(dev_path, encoding="utf-8")
test  = pd.read_csv(test_path, encoding="utf-8")

In [6]:
# Limpieza mínima de caracteres raros (apóstrofo 0x92)
def clean_text(s: str) -> str:
    if not isinstance(s, str):
        return s
    return s.replace('\x92', "")#

for df_ in (train, dev, test):
    df_["Utterance"] = df_["Utterance"].apply(clean_text)


In [7]:
# Unificar splits y quedarnos con las columnas relevantes

##%%
df = pd.concat(
    [
        train.assign(split="train"),
        dev.assign(split="dev"),
        test.assign(split="test"),
    ],
    ignore_index=True,
)

df = df[
    [
        "Dialogue_ID",
        "Utterance_ID",
        "Speaker",
        "Utterance",
        "Emotion",
        "Sentiment",
        "split",
    ]
]

print(df.head())
print(df.shape)


   Dialogue_ID  Utterance_ID          Speaker  \
0            0             0         Chandler   
1            0             1  The Interviewer   
2            0             2         Chandler   
3            0             3  The Interviewer   
4            0             4         Chandler   

                                           Utterance   Emotion Sentiment  \
0  also I was the point person on my companys tra...   neutral   neutral   
1                    You mustve had your hands full.   neutral   neutral   
2                            That I did. That I did.   neutral   neutral   
3       So lets talk a little bit about your duties.   neutral   neutral   
4                             My duties?  All right.  surprise  positive   

   split  
0  train  
1  train  
2  train  
3  train  
4  train  
(13708, 7)


In [8]:
# Construir estructura de diálogos ordenados

##%%
dialogues = (
    df.groupby("Dialogue_ID")
      .apply(lambda x: x.sort_values("Utterance_ID").to_dict("records"))
      .to_dict()
)

# Ejemplo de diálogo
dialogues[0][:5]

C:\Users\Haojie\AppData\Local\Temp\ipykernel_38860\503415823.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sort_values("Utterance_ID").to_dict("records"))


[{'Dialogue_ID': 0,
  'Utterance_ID': 0,
  'Speaker': 'Chandler',
  'Utterance': 'also I was the point person on my companys transition from the KL-5 to GR-6 system.',
  'Emotion': 'neutral',
  'Sentiment': 'neutral',
  'split': 'train'},
 {'Dialogue_ID': 0,
  'Utterance_ID': 0,
  'Speaker': 'Mark',
  'Utterance': 'Why do all youre coffee mugs have numbers on the bottom?',
  'Emotion': 'surprise',
  'Sentiment': 'positive',
  'split': 'test'},
 {'Dialogue_ID': 0,
  'Utterance_ID': 0,
  'Speaker': 'Phoebe',
  'Utterance': 'Oh my God, hes lost it. Hes totally lost it.',
  'Emotion': 'sadness',
  'Sentiment': 'negative',
  'split': 'dev'},
 {'Dialogue_ID': 0,
  'Utterance_ID': 1,
  'Speaker': 'The Interviewer',
  'Utterance': 'You mustve had your hands full.',
  'Emotion': 'neutral',
  'Sentiment': 'neutral',
  'split': 'train'},
 {'Dialogue_ID': 0,
  'Utterance_ID': 1,
  'Speaker': 'Monica',
  'Utterance': 'What?',
  'Emotion': 'surprise',
  'Sentiment': 'negative',
  'split': 'dev'}]

In [9]:
# Normalizar hablantes (SPEAKER_0, SPEAKER_1, ... por diálogo)

##%%
def normalize_speakers(dialogue):
    speakers = list({turn["Speaker"] for turn in dialogue})
    mapping = {spk: f"SPEAKER_{i}" for i, spk in enumerate(speakers)}
    for turn in dialogue:
        turn["Speaker_Norm"] = mapping[turn["Speaker"]]
    return dialogue

dialogues_norm = {
    d_id: normalize_speakers(dialog)
    for d_id, dialog in dialogues.items()
}

dialogues_norm[0][:5]


[{'Dialogue_ID': 0,
  'Utterance_ID': 0,
  'Speaker': 'Chandler',
  'Utterance': 'also I was the point person on my companys transition from the KL-5 to GR-6 system.',
  'Emotion': 'neutral',
  'Sentiment': 'neutral',
  'split': 'train',
  'Speaker_Norm': 'SPEAKER_4'},
 {'Dialogue_ID': 0,
  'Utterance_ID': 0,
  'Speaker': 'Mark',
  'Utterance': 'Why do all youre coffee mugs have numbers on the bottom?',
  'Emotion': 'surprise',
  'Sentiment': 'positive',
  'split': 'test',
  'Speaker_Norm': 'SPEAKER_1'},
 {'Dialogue_ID': 0,
  'Utterance_ID': 0,
  'Speaker': 'Phoebe',
  'Utterance': 'Oh my God, hes lost it. Hes totally lost it.',
  'Emotion': 'sadness',
  'Sentiment': 'negative',
  'split': 'dev',
  'Speaker_Norm': 'SPEAKER_5'},
 {'Dialogue_ID': 0,
  'Utterance_ID': 1,
  'Speaker': 'The Interviewer',
  'Utterance': 'You mustve had your hands full.',
  'Emotion': 'neutral',
  'Sentiment': 'neutral',
  'split': 'train',
  'Speaker_Norm': 'SPEAKER_0'},
 {'Dialogue_ID': 0,
  'Utterance_ID':

In [10]:
# Construir filas con ventana de contexto (por ejemplo 2 turnos anteriores)

##%%
def build_context_rows(dialogue, window=4):
    rows = []
    dialogue = sorted(dialogue, key=lambda x: x["Utterance_ID"])
    for i, turn in enumerate(dialogue):
        ctx = dialogue[max(0, i - window):i]
        ctx_text = " ".join(
            f'{t["Speaker_Norm"]}: {t["Utterance"]}' for t in ctx
        )
        rows.append(
            {
                "dialogue_id": turn["Dialogue_ID"],
                "utt_id": turn["Utterance_ID"],
                "speaker": turn["Speaker_Norm"],
                "context": ctx_text,
                "text": turn["Utterance"],
                "emotion": turn["Emotion"],
                "sentiment": turn["Sentiment"],
                "split": turn["split"],
            }
        )
    return rows

all_rows = []
for d_id, dialog in dialogues_norm.items():
    all_rows.extend(build_context_rows(dialog, window=4))

df_model = pd.DataFrame(all_rows)
df_model.head()

,dialogue_id,utt_id,speaker,context,text,emotion,sentiment,split
0,0,0,SPEAKER_4,,also I was the point person on my companys tra...,neutral,neutral,train
1,0,0,SPEAKER_1,SPEAKER_4: also I was the point person on my c...,Why do all youre coffee mugs have numbers on t...,surprise,positive,test
2,0,0,SPEAKER_5,SPEAKER_4: also I was the point person on my c...,"Oh my God, hes lost it. Hes totally lost it.",sadness,negative,dev
3,0,1,SPEAKER_0,SPEAKER_4: also I was the point person on my c...,You mustve had your hands full.,neutral,neutral,train
4,0,1,SPEAKER_3,SPEAKER_4: also I was the point person on my c...,What?,surprise,negative,dev


In [11]:
# Construir el campo final para el prompt: contexto + [SEP] + turno actual

##%%
df_model["context"] = df_model["context"].fillna("").astype(str)

df_model["input_text"] = (
    "Context:\n" + df_model["context"].str.strip() +
    "\n\nCurrent utterance:\n" + df_model["text"].astype(str).str.strip()
)


df_model[["dialogue_id", "utt_id", "speaker", "input_text", "emotion", "split"]].head(10)


,dialogue_id,utt_id,speaker,input_text,emotion,split
0,0,0,SPEAKER_4,Context:\n\n\nCurrent utterance:\nalso I was t...,neutral,train
1,0,0,SPEAKER_1,Context:\nSPEAKER_4: also I was the point pers...,surprise,test
2,0,0,SPEAKER_5,Context:\nSPEAKER_4: also I was the point pers...,sadness,dev
3,0,1,SPEAKER_0,Context:\nSPEAKER_4: also I was the point pers...,neutral,train
4,0,1,SPEAKER_3,Context:\nSPEAKER_4: also I was the point pers...,surprise,dev
5,0,1,SPEAKER_2,Context:\nSPEAKER_1: Why do all youre coffee m...,anger,test
6,0,2,SPEAKER_2,"Context:\nSPEAKER_5: Oh my God, hes lost it. H...",neutral,test
7,0,2,SPEAKER_4,Context:\nSPEAKER_0: You mustve had your hands...,neutral,train
8,0,3,SPEAKER_0,Context:\nSPEAKER_3: What? SPEAKER_2: Oh. That...,neutral,train
9,0,4,SPEAKER_4,Context:\nSPEAKER_2: Oh. Thats so Monica can k...,surprise,train


In [12]:
# Lista de emociones (clases) presentes en MELD

##%%
emotions = sorted(df_model["emotion"].unique())
print("Emociones:", emotions)

# Separar splits (útil para evaluar solo en test)
train_df = df_model[df_model["split"] == "train"].reset_index(drop=True)
dev_df   = df_model[df_model["split"] == "dev"].reset_index(drop=True)
test_df  = df_model[df_model["split"] == "test"].reset_index(drop=True)

print(train_df["emotion"].value_counts())

Emociones: ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']
emotion
neutral     4710
joy         1743
surprise    1205
anger       1109
sadness      683
disgust      271
fear         268
Name: count, dtype: int64


# Prompt engineering c

A continuación se evalúa FLAN-T5 con **few-shot prompting balanceado** y **label scoring** (se puntúa cada etiqueta candidata y se elige la de menor pérdida).

In [13]:
import torch
import random
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sklearn.metrics import classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Modelo (sin fine-tuning)
MODEL_NAME = "google/flan-t5-base"
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
model.eval()
tok.truncation_side = "left"   # clave: conserva el final (donde está el input real)
MAX_INPUT_LEN = 512            # flan-t5-base suele aguantar 512 mejor que 384


# Etiquetas (clases)
LABELS = [e.lower() for e in emotions]
LABELS_STR = ", ".join(LABELS)

# Ajustes de prompting (tuneables)
SHOTS_PER_LABEL = 1     # prueba 3-4 si no hay truncation
SEED = 42


def build_fewshot(train_df, shots_per_label=SHOTS_PER_LABEL, seed=SEED):
    """Construye un bloque few-shot balanceado (mismos shots por clase) usando ejemplos reales del train."""
    rnd = random.Random(seed)
    blocks = []
    for lab in LABELS:
        subset = train_df[train_df["emotion"].str.lower() == lab]
        if subset.empty:
            continue
        idxs = list(subset.index)
        rnd.shuffle(idxs)
        for i in idxs[:shots_per_label]:
            x = subset.loc[i, "text"]
            blocks.append(f"Example:\n{x}\nLabel: {lab}\n")
    rnd.shuffle(blocks)
    return "\n".join(blocks)

FEWSHOT = build_fewshot(train_df, shots_per_label=SHOTS_PER_LABEL, seed=SEED)

def build_prompt(input_text: str) -> str:
    return (
        "You are a dialogue emotion classifier.\n"
        f"Choose exactly ONE label from: {LABELS_STR}\n"
        "Return only the label (one word), nothing else.\n\n"
        f"{FEWSHOT}\n"
        "Now classify the following dialogue turn:\n"
        f"{input_text}\n"
        "Label:"
    )

# Pre-tokenización de etiquetas (para acelerar el scoring)
pad_id = tok.pad_token_id
_label_ids = tok(LABELS, return_tensors="pt", padding=True, truncation=True).input_ids.to(device)
_decoder_input_ids = model.prepare_decoder_input_ids_from_labels(_label_ids)

@torch.inference_mode()
def score_labels(prompt: str) -> list[float]:
    """Devuelve la loss media por etiqueta (menor = mejor)."""
    enc = tok(prompt, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN).to(device)
    B = _label_ids.size(0)

    out = model(
        input_ids=enc.input_ids.repeat(B, 1),
        attention_mask=enc.attention_mask.repeat(B, 1),
        decoder_input_ids=_decoder_input_ids,
    )
    logits = out.logits  # [B, T, V]
    V = logits.size(-1)

    # Cross-entropy por token, ignorando padding de las etiquetas
    loss_tok = F.cross_entropy(
        logits.view(-1, V),
        _label_ids.view(-1),
        ignore_index=pad_id,
        reduction="none",
    ).view(B, _label_ids.size(1))

    denom = (_label_ids != pad_id).sum(dim=1).clamp(min=1)
    loss_per_label = loss_tok.sum(dim=1) / denom
    return loss_per_label.detach().cpu().tolist()

@torch.inference_mode()
def predict_one(input_text: str) -> str:
    prompt = build_prompt(input_text)
    losses = score_labels(prompt)
    best_i = int(min(range(len(losses)), key=lambda i: losses[i]))
    return LABELS[best_i]

def predict_batch(texts: list[str]) -> list[str]:
    return [predict_one(t) for t in texts]

Device: cuda


In [14]:
# Evaluación en el conjunto de test (muestra)
sample_size = 2000
test_sample = test_df.sample(n=min(sample_size, len(test_df)), random_state=42)

texts = test_sample["input_text"].tolist()
y_true = test_sample["emotion"].str.lower().tolist()
y_pred = predict_batch(texts)

print(
    classification_report(
        y_true,
        y_pred,
        labels=LABELS,
        target_names=LABELS,
        zero_division=0,
    )
)

              precision    recall  f1-score   support

       anger       0.23      0.25      0.24       270
     disgust       0.18      0.06      0.09        47
        fear       0.05      0.03      0.04        35
         joy       0.37      0.18      0.24       301
     neutral       0.64      0.26      0.37       982
     sadness       0.22      0.33      0.26       157
    surprise       0.16      0.68      0.25       208

    accuracy                           0.28      2000
   macro avg       0.26      0.25      0.21      2000
weighted avg       0.44      0.28      0.30      2000



## Fine-tuning de FLAN‑T5 para clasificación (Seq2Seq)

La sección anterior usa **prompting + scoring** sin entrenamiento. Aquí se añade **fine‑tuning supervisado** para que el modelo aprenda a producir la etiqueta correcta.

**Idea:** el modelo recibe el mismo `input_text` (Context + Current utterance) y aprende a generar como salida una de las etiquetas:  
`anger, disgust, fear, joy, neutral, sadness, surprise`.

> Si entrenas en CPU o con poca VRAM, activa `USE_LORA = True` (entrena pocos parámetros).

In [15]:
# (Opcional) Instala/actualiza dependencias si tu entorno lo requiere
# !pip -q install -U transformers datasets accelerate sentencepiece evaluate peft

In [16]:
from datasets import Dataset

# Etiquetas fijas (por claridad)
LABELS = ["anger","disgust","fear","joy","neutral","sadness","surprise"]

def make_source(input_text: str) -> str:
    return (
        "Classify the emotion of the Current utterance given the Context.\n"
        f"Return exactly one label from: {', '.join(LABELS)}.\n\n"
        f"{input_text}\n\n"
        "Label:"
    )

# DataFrames -> Datasets
train_tmp = train_df[["input_text","emotion"]].copy()
dev_tmp   = dev_df[["input_text","emotion"]].copy()
test_tmp  = test_df[["input_text","emotion"]].copy()

train_tmp["emotion"] = train_tmp["emotion"].str.lower().str.strip()
dev_tmp["emotion"]   = dev_tmp["emotion"].str.lower().str.strip()
test_tmp["emotion"]  = test_tmp["emotion"].str.lower().str.strip()

train_ds = Dataset.from_pandas(train_tmp, preserve_index=False)
dev_ds   = Dataset.from_pandas(dev_tmp, preserve_index=False)
test_ds  = Dataset.from_pandas(test_tmp, preserve_index=False)

len(train_ds), len(dev_ds), len(test_ds)

(9989, 1109, 2610)

In [17]:
import torch
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
)

FT_MODEL_NAME = "google/flan-t5-base"

tok_ft = AutoTokenizer.from_pretrained(FT_MODEL_NAME)
tok_ft.truncation_side = "left"  # conserva el final (donde suele estar el Current utterance)

model_ft = AutoModelForSeq2SeqLM.from_pretrained(FT_MODEL_NAME)

MAX_SOURCE_LEN = 512
MAX_TARGET_LEN = 8  # las etiquetas son cortas

def preprocess(batch):
    sources = [make_source(t) for t in batch["input_text"]]
    targets = batch["emotion"]

    model_inputs = tok_ft(sources, max_length=MAX_SOURCE_LEN, truncation=True)
    with tok_ft.as_target_tokenizer():
        labels = tok_ft(targets, max_length=MAX_TARGET_LEN, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tok = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
dev_tok   = dev_ds.map(preprocess, batched=True, remove_columns=dev_ds.column_names)
test_tok  = test_ds.map(preprocess, batched=True, remove_columns=test_ds.column_names)

data_collator = DataCollatorForSeq2Seq(tokenizer=tok_ft, model=model_ft)

Map:   0%|          | 0/9989 [00:00<?, ? examples/s]C:\Users\Haojie\Desktop\MacOS\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
Map: 100%|██████████| 2610/2610 [00:00<00:00, 9050.05 examples/s]


In [18]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
from sklearn.metrics import accuracy_score, f1_score, classification_report

def norm_label(s: str) -> str:
    s = (s or "").strip().lower()
    s = s.split()[0] if s else ""
    s = s.strip(".,:;!?'\"()[]{}")
    return s if s in LABELS else "unknown"

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]

    pred_text = tok_ft.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tok_ft.pad_token_id)
    label_text = tok_ft.batch_decode(labels, skip_special_tokens=True)

    y_pred = [norm_label(x) for x in pred_text]
    y_true = [norm_label(x) for x in label_text]

    acc = accuracy_score(y_true, y_pred)
    f1m = f1_score(y_true, y_pred, average="macro", labels=LABELS)
    f1w = f1_score(y_true, y_pred, average="weighted", labels=LABELS)
    return {"accuracy": acc, "f1_macro": f1m, "f1_weighted": f1w}

# ====== LoRA (recomendado si NO tienes GPU / poca VRAM) ======
USE_LORA = False  # pon True si entrenas en CPU o te falta VRAM

if USE_LORA:
    from peft import LoraConfig, get_peft_model, TaskType

    lora_cfg = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        # Para T5: "q" y "v" suelen funcionar bien
        target_modules=["q", "v"],
    )
    model_ft = get_peft_model(model_ft, lora_cfg)
    model_ft.print_trainable_parameters()

ValueError: Your currently installed version of Keras is Keras 3, but this is not yet supported in Transformers. Please install the backwards-compatible tf-keras package with `pip install tf-keras`.

In [ ]:
# Ajusta estos hiperparámetros según tu hardware
# - Si tienes GPU: batch 8-16 suele ir bien
# - Si CPU: baja batch y activa USE_LORA arriba
training_args = Seq2SeqTrainingArguments(
    output_dir="flan_meld_finetuned",
    learning_rate=2e-5 if not USE_LORA else 2e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    evaluation_strategy="steps",
    eval_steps=500,
    save_steps=500,
    logging_steps=100,
    predict_with_generate=True,
    generation_max_length=8,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
)

trainer = Seq2SeqTrainer(
    model=model_ft,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=dev_tok,
    data_collator=data_collator,
    tokenizer=tok_ft,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
# Evaluación final en test
pred = trainer.predict(test_tok)
pred_text = tok_ft.batch_decode(pred.predictions, skip_special_tokens=True)

y_true = test_tmp["emotion"].tolist()
y_pred = [norm_label(x) for x in pred_text]

print(classification_report(y_true, y_pred, labels=LABELS, target_names=LABELS, zero_division=0))

In [ ]:
# Guardar (modelo + tokenizer)
trainer.save_model("flan_meld_finetuned/best_model")
tok_ft.save_pretrained("flan_meld_finetuned/best_model")

# Para recargar después:
# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
# tok_ft = AutoTokenizer.from_pretrained("flan_meld_finetuned/best_model")
# model_ft = AutoModelForSeq2SeqLM.from_pretrained("flan_meld_finetuned/best_model")

## Notas

- Este notebook incluye:
  - **Prompt engineering** con FLAN‑T5 (sin entrenamiento).
  - **Fine‑tuning supervisado** (Seq2SeqTrainer) para mejorar métricas.
- Si el prompt crece mucho, mantén `tok.truncation_side = "left"` y `MAX_INPUT_LEN/MAX_SOURCE_LEN = 512` para conservar el **Current utterance**.
- Si entrenas sin GPU o con poca VRAM, activa `USE_LORA = True` en la sección de fine‑tuning.